# Food Delivery Marketplace Analytics
## 03 — Cohort Retention & Lifetime Value Analysis

Track customer retention and repeat purchase patterns across cohorts.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path().resolve().parent
sys.path.insert(0, str(BASE_DIR))

from src.db_connect import get_engine

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

print("✅ Environment initialized")

## 1. Load Data & Build Cohort Table

In [ ]:
engine, dialect = get_engine()

with engine.connect() as conn:
    df_orders = pd.read_sql("SELECT * FROM fact_orders", conn)
    df_customers = pd.read_sql("SELECT * FROM dim_customers", conn)
    df_order_items = pd.read_sql("SELECT * FROM fact_order_items", conn)

# Prepare data
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders = df_orders.merge(df_customers[['customer_id', 'customer_unique_id', 'cohort_month']], 
                             on='customer_id', how='left')

print(f"Orders: {len(df_orders):,}")
print(f"Unique Customers: {df_customers['customer_unique_id'].nunique():,}")

## 2. Cohort Retention Analysis

Measure what % of customers from each cohort return in subsequent months.

In [ ]:
# Add order month
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.strftime('%Y-%m')

# Get customer cohort first order month
customer_cohorts = df_orders.groupby('customer_unique_id').agg({
    'order_month': 'min'
}).reset_index()
customer_cohorts.columns = ['customer_unique_id', 'cohort']

# Merge back to orders
df_cohort = df_orders.merge(customer_cohorts, on='customer_unique_id', how='left')

# Calculate months since cohort
df_cohort['cohort_date'] = pd.to_datetime(df_cohort['cohort'])
df_cohort['order_date'] = pd.to_datetime(df_cohort['order_month'])
df_cohort['months_since_cohort'] = (
    (df_cohort['order_date'].dt.year - df_cohort['cohort_date'].dt.year) * 12 + 
    (df_cohort['order_date'].dt.month - df_cohort['cohort_date'].dt.month)
)

# Cohort table: cohort vs months_since_cohort
cohort_pivot = df_cohort.groupby(['cohort', 'months_since_cohort'])['customer_unique_id'].nunique().unstack(fill_value=0)

print("Cohort Sizes (First Month Customers):")
print(cohort_pivot.iloc[:, 0])

# Convert to retention rates
cohort_retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0) * 100
cohort_retention = cohort_retention.round(1)

print("\nCohort Retention Matrix (%):")
print(cohort_retention.head(10))

## 3. Cohort Heatmap Visualization

In [ ]:
# Visualize as heatmap
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(cohort_retention, annot=True, fmt='.0f', cmap='RdYlGn', 
            cbar_kws={'label': 'Retention %'}, ax=ax, vmin=0, vmax=100,
            linewidths=0.5, linecolor='gray')

ax.set_title('Cohort Retention Rates (%) — Heat Map', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Months Since Cohort Start', fontsize=11, fontweight='bold')
ax.set_ylabel('Cohort Month', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / '07_cohort_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Heatmap saved to reports/07_cohort_retention_heatmap.png")

## 4. Lifetime Value (LTV) by Cohort

Total GMV per customer across all their orders by cohort.

In [ ]:
# Add GMV per order
order_gmv = df_order_items.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum'
}).reset_index()
order_gmv['gmv'] = order_gmv['price'] + order_gmv['freight_value']

df_cohort = df_cohort.merge(order_gmv[['order_id', 'gmv']], on='order_id', how='left')

# Filter delivered orders only
df_cohort_delivered = df_cohort[df_cohort['order_status'] == 'delivered'].copy()

# LTV by cohort
ltv_by_cohort = df_cohort_delivered.groupby('cohort').agg({
    'customer_unique_id': 'nunique',
    'gmv': ['sum', 'mean'],
    'order_id': 'count'
}).round(2)

ltv_by_cohort.columns = ['customers', 'total_revenue', 'ltv_per_customer', 'total_orders']
ltv_by_cohort = ltv_by_cohort.reset_index()
ltv_by_cohort['orders_per_customer'] = (ltv_by_cohort['total_orders'] / ltv_by_cohort['customers']).round(2)

print("Lifetime Value by Customer Cohort (Delivered Orders):")
print(ltv_by_cohort.to_string(index=False))